In [3]:
"""
Notebook: data_preprocessing.ipynb
Objetivo: Limpieza, transformación y preprocesamiento de los datos
del dataset BRFSS2015 del CDC para el posterior entrenamiento de
modelos de ML en detección de diabetes.

Autor: Jesús Rodríguez
Fecha: 06/11/2025
"""

'\nNotebook: data_preprocessing.ipynb\nObjetivo: Limpieza, transformación y preprocesamiento de los datos\ndel dataset BRFSS2015 del CDC para el posterior entrenamiento de\nmodelos de ML en detección de diabetes.\n\nAutor: Jesús Rodríguez\nFecha: 06/11/2025\n'

## 1. Configuración inicial

In [4]:
# Librerías básicas
from pathlib import Path
import pandas as pd
import numpy as np


# Interpretabilidad
from IPython.display import display

# Google Drive (solo necesario si se ejecuta en Colab)
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [5]:
# Definición de rutas
DATA_DIR_LOCAL = Path("data")
DATA_FILE = "raw_dataset.csv"
DATA_PATH = DATA_DIR_LOCAL / DATA_FILE

# Montaje de Google Drive (solo en Colab)
if IN_COLAB and not DATA_PATH.exists():
    drive.mount("/content/drive")
    data_dir = Path("/content/drive/MyDrive/TFM/data")
    DATA_PATH = data_dir / DATA_FILE

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Verificación del dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo '{DATA_FILE}'. "
        f"Esperado en: {DATA_PATH}"
    )

# Carga del dataset
df_raw = pd.read_csv(
    DATA_PATH,
    encoding="latin-1"
)

In [ ]:
# Información general del dataset
print(f"Shape del dataset: {df_raw.shape}")

df_raw.info()
df_raw.describe().transpose()

## 2. Análisis exploratorio inicial del target

In [ ]:
# Distribución absoluta y relativa del target
target_counts = df_raw['DIABETE3'].value_counts()
target_pct = df_raw['DIABETE3'].value_counts(normalize=True).mul(100)

target_distribution = (  # pylint: disable=C0103
    pd.DataFrame({
        'Conteo': target_counts,
        'Porcentaje (%)': target_pct
    })
    .sort_index()
)
# Se deshabilita C0103 porque target_distribution no sigue UPPER_CASE,
# pero es más legible en análisis de datos.

print("Distribución de clases:")
print(target_distribution)

## 3. Cribado de columnas

In [ ]:
# Se conservan únicamente las clases relevantes del target
TARGET_CLASSES = [1, 3]

df_raw = (
    df_raw[df_raw['DIABETE3'].isin(TARGET_CLASSES)]
    .reset_index(drop=True)
    .copy()
)

### 3.1. Primer cribado de columnas (330 variables)

In [ ]:
# Eliminación de registros duplicados
df_raw = df_raw.drop_duplicates().copy()

In [ ]:
# Definición de columnas irrelevantes para la detección de diabetes
COLS_TO_DROP_STAGE_1 = [
    '_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO',
    '_PSU', 'QSTVER', 'QSTLANG', 'MSCODE', '_STSTR', '_STRWT', '_RAWRAKE',
    '_WT2RAKE', '_CLLCPWT', '_DUALUSE', '_DUALCOR', '_LLCPWT', 'CTELENUM',
    'CELLFON3', 'CTELNUM1', 'CELLFON2', 'CADULT', 'CCLGHOUS', 'CSTATE',
    'LANDLINE', 'NUMPHON2', 'CPDEMO1', 'INTERNET', 'PVTRESD1', 'COLGHOUS',
    'STATERES', 'LADULT', 'NUMADULT', 'NUMMEN', 'NUMWOMEN', 'PVTRESD2',
    'HHADULT', 'HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'CHECKUP1', 'RENTHOM1',
    'NUMHHOL2', 'CHILDREN', 'INCOME2', 'CDHELP', 'CDSOCIAL', '_CHLDCNT',
    '_INCOMG', 'DIABAGE2', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'CHCKIDNY',
    'PDIABTST', 'PREDIAB1', 'INSULIN', 'FEETCHK2', 'DOCTDIAB', 'CHKHEMO3',
    'FEETCHK', 'EYEEXAM', 'DIABEYE', 'DIABEDU', 'MARITAL', 'EDUCA','VETERAN3',
    'EMPLOY1', 'SEATBELT', 'CDDISCUS', 'SCNTMNY1', 'SCNTMEL1', 'SCNTPAID',
    'SCNTWRK1', 'SCNTLPAD', 'SCNTLWK1', 'SXORIENT', 'TRNSGNDR', 'RCSGENDR',
    'RCSRLTN2', 'EMTSUPRT', 'LSATISFY', 'ADPLEASR', 'ADDOWN', 'MISTMNT',
    '_EDUCAG', '_RFSEAT2', '_RFSEAT3', '_LMTWRK1', '_LMTSCL1', 'CAREGIV1',
    'CRGVREL1', 'CRGVLNG1', 'CRGVHRS1', 'CRGVPRB1', 'CRGVPERS', 'CRGVHOUS',
    'CRGVMST2', 'CRGVEXPT', 'CDHOUSE', 'CDASSIST', 'WEIGHT2', 'HEIGHT3',
    'HTIN4', 'HTM4', 'WTKG3', '_BMI5', '_RFBMI5', 'STOPSMK2', 'LASTSMK2',
    'USENOW3', '_SMOKER3', '_RFSMOK3', 'ALCDAY5', 'DRNK3GE5', 'MAXDRNKS',
    'DRNKANY5', 'DROCDY3_', '_RFBING5', '_DRNKWEK', '_RFDRHV5', 'FVGREEN',
    'ARTHEXER', 'FVORANG', 'VEGETAB1', 'GRENDAY_', 'ORNGDAY_', 'VEGEDA1_',
    '_MISVEGN', '_VEGLT1', '_VEG23', '_VEGETEX', 'FTJUDA1_', 'FRUTDA1_',
    'FRUITJU1', 'FRUIT1', '_MISFRTN', '_FRTLT1', '_FRT16', '_FRUITEX',
    'FVBEANS', 'EXERANY2', 'EXRACT11', 'EXRACT21', 'EXEROFT2', 'EXERHMM2',
    'ADMOVE', 'EXACTOT1', 'EXACTOT2', '_TOTINDA', 'METVL11_', 'METVL21_',
    'MAXVO2_', 'FC60_', 'ACTIN11_', 'ACTIN21_', 'PADUR1_', 'PADUR2_',
    'PAFREQ1_', 'PAFREQ2_', '_MINAC11', '_MINAC21', 'STRFREQ_', 'PAMIN11_',
    'PAMIN21_', 'PA1MIN_', 'PAVIG11_', 'PAVIG21_', 'PA1VIGM_', '_PA150R2',
    '_PA300R2', '_PA30021', '_PASTRNG', '_PAREC1', '_PASTAE1', '_LMTACT1',
    'LMTJOIN3', 'ARTHSOCL', 'JOINPAIN', 'PAINACT2', 'TOLDHI2', 'QLMENTL2',
    'QLSTRES2', 'QLHLTH2', '_RFHLTH', 'VIDFCLT2', 'VIREDIF3', 'VIPRFVS2',
    'VINOCRE2', 'VIEYEXM2', 'VIINSUR2', 'VICTRCT4', 'VIGLUMA2', 'VIMACDG2',
    'ASTHMAGE', 'ASATTACK', 'ASERVIST', 'ASDRVIST', 'ASRCHKUP', 'ASACTLIM',
    'ASYMPTOM', 'ASNOSLEP', 'ASTHMED3', 'ASINHALR', 'CASTHDX2', 'CASTHNO2',
    '_LTASTH1', '_CASTHM1', '_ASTHMS1', 'HAREHAB1', 'STREHAB1', 'CVDASPRN',
    'ASPUNSAF', '_MICHD', 'RLIVPAIN', 'RDUCHART', 'RDUCSTRK', 'ARTTODAY',
    'ARTHWGT', 'ARTHEDU', '_DRDXAR1', '_RFCHOL',  '_CHISPNC', '_CRACE1',
    '_CPRACE', '_PRACE1', '_MRACE1', '_HISPANC', '_RACEG21', '_RACEGR3',
    '_RACE_G1', '_AGE65YR', '_AGE80', '_AGE_G', '_RFHYPE5', 'CHOLCHK',
    'HIVTST6', 'HIVTSTD3', 'WHRTST10', 'HADMAM', 'HOWLONG', 'HADPAP2',
    'LASTPAP2', 'HPVTEST', 'HPLSTTST', 'HADHYST2', 'PROFEXAM', 'LENGEXAM',
    'BLDSTOOL', 'LSTBLDS3', 'HADSIGM3', 'HADSGCO1', 'LASTSIG3', 'PSATEST1',
    'PSATIME', 'PCPSARS1', 'PCPSADE1', 'PCDMDECN', '_AIDTST3', '_CHOLCHK',
    'FLUSHOT6', 'FLSHTMY2', 'IMFVPLAC', 'PNEUVAC3', 'TETANUS', 'HPVADVC2',
    'HPVADSHT', 'SHINGLE2', '_FLSHOT6', '_PNEUMO2', 'DRADVISE', 'PREGNANT',
    'PCPSAAD2', 'PCPSADI1', 'PCPSARE1', '_HCVU651', 'STRENGTH', 'PAMISS1_',
    'SMOKDAY2', 'EXERHMM1', 'WTCHSALT', 'LONGWTCH', 'BEANDAY_', '_FRTRESP',
    '_VEGRESP', '_PAINDX1', 'CIMEMLOS', 'ASTHMA3', 'ASTHNOW', 'CHCSCNCR',
    'CHCOCNCR', 'CHCCOPD1', 'BLDSUGAR'
]

In [ ]:
# Eliminación de columnas irrelevantes
df_cleaned_1 = (
    df_raw
    .drop(columns=COLS_TO_DROP_STAGE_1, errors='ignore')
    .copy()
)

In [ ]:
# Comprobación de consistencia del número total de columnas
total_columns = df_cleaned_1.shape[1] + len(COLS_TO_DROP_STAGE_1)
print(f'Total de columnas consideradas en el cribado: {total_columns}')

### 3.2. Segundo cribado de columnas (34 variables)

In [ ]:
# Cálculo del número y porcentaje de valores nulos por columna
nulls = (
    df_cleaned_1
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

nulls_pct = (
    df_cleaned_1
    .isnull()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

nulls_summary = pd.DataFrame({
    'Nulos': nulls,
    'Porcentaje (%)': nulls_pct
})

display(nulls_summary)

In [ ]:
# Columnas con elevado número de NaN o menor relevancia clínica
COLS_TO_DROP_STAGE_2 = [
    'ADSLEEP', 'ADENERGY', 'ADEAT1', 'ADFAIL',
    'ADTHINK', 'ADANXEV', 'ARTHDIS2',
    'POORHLTH', 'AVEDRNK2'
    ]

In [ ]:
# Eliminación de columnas seleccionadas
df_cleaned_2 = (
    df_cleaned_1
    .drop(columns=COLS_TO_DROP_STAGE_2, errors='ignore')
    .copy()
)

In [ ]:
# Comprobación de consistencia del número total de columnas
total_columns = df_cleaned_2.shape[1] + len(COLS_TO_DROP_STAGE_2)
print(f'Total de columnas consideradas en el cribado: {total_columns}')

### 3.3. Tercer cribado de columnas (25 variables)

In [ ]:
class MissingValueAnalyzer:
    """
    Analiza porcentajes de valores especiales en un DataFrame, que representan
    respuestas inválidas o no informadas.
    """

    def __init__(self, df: pd.DataFrame):
        """Inicializa con un DataFrame."""
        self.df = df

    def calculate_percentage(
        self, columns: list[str], special_values: list
    ) -> pd.DataFrame:
        """
        Calcula el porcentaje de valores especiales por columna.

        Args:
            columns: Columnas a analizar.
            special_values: Valores considerados especiales.

        Returns:
            DataFrame con columnas 'Columna' y 'Porcentaje (%)'.
        """
        results = [
            {
                "Columna": col,
                "Porcentaje (%)": (self.df[col].isin(special_values).sum() /
                                   max(self.df[col].notna().sum(), 1) * 100)
            }
            for col in columns
        ]
        return pd.DataFrame(results)

    def calculate_all_groups(self, column_groups: dict) -> dict:
        """
        Calcula porcentajes para varios grupos de columnas.

        Args:
            column_groups: dict {nombre_grupo: (columnas, valores_especiales)}

        Returns:
            dict con DataFrames como valores.
        """
        return {
            name: self.calculate_percentage(cols, values)
            for name, (cols, values) in column_groups.items()
        }

In [ ]:
# Creación del analizador de valores faltantes
missing_value_analyzer = MissingValueAnalyzer(df_cleaned_2)

In [ ]:
# Definición de grupos de columnas y valores especiales
COLUMN_GROUPS = {
    '7_9': (
        ['GENHLTH', 'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3',
         'QLACTLM2', 'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK',
         'DIFFDRES', 'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_PACAT1'],
        [7, 9]
    ),
    '77_88_99': (
        ['PHYSHLTH', 'MENTHLTH'],
        [77, 88, 99]
    ),
    '777_999': (
        ['EXEROFT1'],
        [777, 999]
    ),
    '9': (
        ['_RACE'],
        [9]
    ),
    '14': (
        ['_AGEG5YR'],
        [14]
    )
}

In [ ]:
# Cálculo de resultados por grupo
missing_value_results = missing_value_analyzer.calculate_all_groups(COLUMN_GROUPS)

In [ ]:
# Visualización de resultados
for group_name, df_result in missing_value_results.items():
    print(f"\n=== Resultados para grupo {group_name} ===")
    print(df_result)

In [ ]:
# Eliminación de columnas con valores especiales significativos
COLS_TO_DROP_STAGE_3 = ['PHYSHLTH', 'MENTHLTH']

df_cleaned_3 = (
    df_cleaned_2
    .drop(columns=COLS_TO_DROP_STAGE_3, errors='ignore')
    .copy()
)

In [ ]:
# Comprobaciones finales de consistencia
total_columns = df_cleaned_3.shape[1] + len(COLS_TO_DROP_STAGE_3)

print(f'Total de columnas consideradas en el cribado: {total_columns}')
print(f'Número final de columnas: {df_cleaned_3.shape[1]}')

## 4. Transformación e imputación de variables

### 4.1. Variables que podrían estar relacionadas

#### BPMEDS y su relación con BPHIGH

In [ ]:
# Indicador auxiliar de valores nulos en BPMEDS
df_cleaned_3['BPMEDS_is_null'] = df_cleaned_3['BPMEDS'].isna()

In [ ]:
# Distribución de nulos de BPMEDS según hipertensión diagnosticada
bphigh_distribution = pd.crosstab(
    df_cleaned_3['BPMEDS_is_null'],
    df_cleaned_3['BPHIGH4'],
    margins=True,
    normalize='columns'
)

print("Distribución de BPMEDS nulos según BPHIGH4:")
print(bphigh_distribution)

In [ ]:
# Imputación clínica:
# Si no tiene hipertensión, no toma medicación → categoría 2 (NO)
df_cleaned_3['BPMEDS'] = df_cleaned_3['BPMEDS'].fillna(2)

In [ ]:
# Verificación de imputación
print(f"Número de NaN restantes en BPMEDS: {df_cleaned_3['BPMEDS'].isna().sum()}")

In [ ]:
# Eliminación de la variable auxiliar
df_cleaned_3.drop(columns='BPMEDS_is_null', inplace=True)

#### EXEROFT1 y limitaciones de movilidad

In [ ]:
# Indicador auxiliar de valores nulos en EXEROFT1
df_cleaned_3['EXEROFT1_is_null'] = df_cleaned_3['EXEROFT1'].isna()

In [ ]:
# Análisis de relación entre nulos en EXEROFT1 y limitaciones funcionales
for variable in ['DIFFALON', 'DIFFDRES', 'DIFFWALK']:
    crosstab = pd.crosstab(
        df_cleaned_3['EXEROFT1_is_null'],
        df_cleaned_3[variable],
        margins=True,
        normalize='columns'
    )

    print(f"\nDistribución de nulos de EXEROFT1 según {variable}:")
    print(crosstab)

In [ ]:
# Conclusión: los NaN no tienen un significado clínico claro
# Se elimina la variable auxiliar
df_cleaned_3.drop(columns='EXEROFT1_is_null', inplace=True)

### 4.2. Tratamiento de variables especiales

#### EXEROFT1 – decodificación de frecuencia de ejercicio

In [ ]:
def decode_exeroft1(value):
    """
    Decodifica la variable EXEROFT1:
    - 101–199 → veces por semana
    - 201–299 → veces por mes (convertido a semana)
    - 777, 999 → No sabe / No contesta (se conservan)
    """
    if pd.isna(value):
        return np.nan

    if 101 <= value <= 199:
        return value - 100

    if 201 <= value <= 299:
        return (value - 200) / (30.0 / 7.0)

    if value in (777, 999):
        return value

    return np.nan

In [ ]:
# Aplicación de la transformación
df_cleaned_3['EXEROFT1'] = df_cleaned_3['EXEROFT1'].apply(decode_exeroft1)

In [ ]:
# Inspección de valores tras la transformación
df_cleaned_3['EXEROFT1'].round(2).unique()

#### _FRUTSUM y _VEGESUM – normalización de consumo diario

In [ ]:
# Conversión de raciones * 100 a raciones reales por día
df_cleaned_3['_FRUTSUM'] = df_cleaned_3['_FRUTSUM'] / 100
df_cleaned_3['_VEGESUM'] = df_cleaned_3['_VEGESUM'] / 100

In [ ]:
# Verificación de valores transformados
print(df_cleaned_3['_FRUTSUM'].round(2).unique())
print(df_cleaned_3['_VEGESUM'].round(2).unique())

### 4.3. Imputación

#### Eliminación inicial de NaN estructurales

In [ ]:
# Eliminación de registros con NaN reales (estructurales)
df_cleaned_3 = df_cleaned_3.dropna()

In [ ]:
# Verificación del estado del dataset
df_cleaned_3.info()

#### Identificación de valores especiales (“No sabe / Se negó”)

In [ ]:
# Valores especiales que representan respuestas inválidas
# (_BMI5CAT, _FRUTSUM, _VEGESUM y SEX no presentan estas categorías)
# DIABETE3 ya fue filtrada previamente
INVALID_VALUES = {
    'GENHLTH': [7, 9],
    'BPHIGH4': [7, 9],
    'BPMEDS': [7, 9],
    'BLOODCHO': [7, 9],
    'HAVARTH3': [7, 9],
    'QLACTLM2': [7, 9],
    'USEEQUIP': [7, 9],
    'BLIND': [7, 9],
    'DECIDE': [7, 9],
    'DIFFWALK': [7, 9],
    'DIFFDRES': [7, 9],
    'DIFFALON': [7, 9],
    'SMOKE100': [7, 9],
    'ADDEPEV2': [7, 9],
    '_PACAT1': [9],
    '_RACE': [9],
    '_AGEG5YR': [14],
    'EXEROFT1': [777, 999]
}

In [ ]:
# Sustitución de valores especiales por NaN
for column, invalid_values in INVALID_VALUES.items():
    df_cleaned_3[column] = df_cleaned_3[column].replace(invalid_values, np.nan)

#### Separación por tipo de variable

In [ ]:
# Definición de tipos de variables
CATEGORICAL_NOMINAL = [
    'BPHIGH4', 'BPMEDS', 'BLOODCHO', 'HAVARTH3', 'QLACTLM2',
    'USEEQUIP', 'BLIND', 'DECIDE', 'DIFFWALK', 'DIFFDRES',
    'DIFFALON', 'SMOKE100', 'ADDEPEV2', '_RACE'
]

CATEGORICAL_ORDINAL = [
    'GENHLTH', '_PACAT1', '_AGEG5YR'
]

NUMERICAL = [
    'EXEROFT1'
]

#### Imputación

In [ ]:
# Imputación de variables categóricas (nominales y ordinales)
for column in CATEGORICAL_NOMINAL + CATEGORICAL_ORDINAL:
    df_cleaned_3[column] = df_cleaned_3[column].fillna(-1)

In [ ]:
# Imputación de variables numéricas mediante la mediana
for column in NUMERICAL:
    df_cleaned_3[column] = df_cleaned_3[column].fillna(
        df_cleaned_3[column].median()
    )

In [ ]:
# Reindexado final del dataset
df_cleaned_3 = df_cleaned_3.reset_index(drop=True)

## 5. Exportación

In [ ]:
# Verificación final del dataset antes de exportar
df_cleaned_3.info()

In [ ]:
# Exportación del dataset limpio
OUTPUT_PATH = "/content/drive/MyDrive/TFM/data/cleaned_dataset.csv"
df_cleaned_3.to_csv(OUTPUT_PATH, index=False)